# Criblage des 5 architectures — étage 1 du protocole en entonnoir

**Objectif :** éliminer rapidement les architectures les plus faibles avant d'engager la validation croisée complète (étage 2), beaucoup plus coûteuse.

Protocole : **1 seul pli**, 20 époques max, early stopping. Les 3 meilleures passeront à l'étage 2 (5 plis).

## Les deux questions auxquelles ce notebook répond

1. **Quelle architecture retenir par modalité ?** — critère : AUC, avec test de DeLong pour vérifier que les écarts sont réels.
2. **L'hypothèse « plus gros ≠ meilleur » se vérifie-t-elle ?** — Alswilem & Pacal (2025) concluent que ReXNet-200 offre le meilleur compromis performance/efficacité. Sur nos ~450 images d'entraînement, EfficientNet-B7 (64 M paramètres) et ViT-B/16 (86 M) devraient sur-apprendre. Le graphique AUC × paramètres testera cette hypothèse directement.

## Avant de lancer
- GPU activé (T4 suffit), `projet.zip` à jour sur Drive (avec la correction du bruit coloré).
- Durée estimée : **20 à 30 min** sur T4 pour BUSI.

> ⚠️ **EfficientNet-B7 est évalué hors de sa résolution native (600 px).** L'exécuter en 224 px le désavantage. Ce biais est assumé — le coût GPU d'une évaluation à 600 px serait multiplié par ~7 — mais il **doit être mentionné dans le rapport**. Ne concluez pas « B7 est mauvais » mais « B7 à 224 px est mauvais ».

In [ ]:
# ── 1. Drive + décompression vers le disque local ────────────────────────
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import sys, os, zipfile, time, shutil

DRIVE = Path('/content/drive/MyDrive')
zips = sorted(DRIVE.glob('projet*.zip')) or sorted(DRIVE.glob('*breast*multimodal*.zip'))
assert zips, f"Aucun zip du projet trouvé dans {DRIVE}"
ZIP = zips[0]

REPO = Path('/content/breast-multimodal-fusion')
if not REPO.exists():
    t0 = time.time()
    with zipfile.ZipFile(ZIP) as z:
        z.extractall('/content/_unzip')
    cands = list(Path('/content/_unzip').rglob('data/manifest.csv'))
    assert cands, 'manifest.csv introuvable dans le zip'
    cands[0].parents[1].rename(REPO)
    print(f'Décompressé en {time.time()-t0:.0f}s')

os.chdir(REPO); sys.path.insert(0, str(REPO))
CKPT_DIR = DRIVE / 'breast-multimodal-checkpoints'; CKPT_DIR.mkdir(exist_ok=True)
RESULTS_DIR = DRIVE / 'breast-multimodal-results'; RESULTS_DIR.mkdir(exist_ok=True)
print('Dépôt :', REPO)

In [ ]:
# ── 2. Dépendances et GPU ────────────────────────────────────────────────
%pip -q install timm albumentations --no-warn-script-location

import torch, timm
print('torch', torch.__version__, '| timm', timm.__version__)
assert torch.cuda.is_available(), "Exécution → Modifier le type d'exécution → GPU"
GPU = torch.cuda.get_device_name(0)
VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU : {GPU} ({VRAM:.0f} Go)')

In [ ]:
# ── 3. Garde-fous avant tout entraînement ────────────────────────────────
!python -m pytest tests/test_no_patient_leakage.py -q --no-header 2>&1 | tail -3

In [ ]:
# ── 4. Configuration du criblage ─────────────────────────────────────────
# Changer DATASET pour cribler la mammographie ('cbis_ddsm' ou 'cmmd').
# ⚠️ Les mammographies sont bien plus lourdes (jusqu'à 5808×3824) : comptez
#    plusieurs heures. Commencez par l'échographie.
DATASET   = 'busi'
FOLD      = 0
EPOCHS    = 20
PATIENCE  = 5
SEED      = 42

# Taille de lot par architecture : les gros modèles saturent la VRAM d'une T4.
# En cas de dépassement mémoire, la boucle réduit automatiquement de moitié.
BATCH = {
    'resnet18':        32,
    'rexnet_200':      32,
    'swin_tiny':       32,
    'vit_b16':         24,
    'efficientnet_b7': 12,   # 64 M paramètres — le plus gourmand
}
ARCHITECTURES = list(BATCH)
print('À cribler :', ', '.join(ARCHITECTURES))

In [ ]:
# ── 5. Données ───────────────────────────────────────────────────────────
import pandas as pd, yaml
from torch.utils.data import DataLoader
from src.data.datasets import BreastDataset, build_transforms
from src.data.splits import assert_no_patient_leakage

cfg = yaml.safe_load(open('configs/data.yaml'))
manifest = pd.read_csv('data/manifest.csv')

sub = manifest[(manifest.dataset == DATASET) & (manifest.fold >= 0)]
train_df = sub[sub.fold != FOLD].reset_index(drop=True)
val_df   = sub[sub.fold == FOLD].reset_index(drop=True)
assert_no_patient_leakage({'train': train_df, 'val': val_df})

ROOT = REPO / 'data/raw'
tf_train = build_transforms(cfg, train=True)
tf_val   = build_transforms(cfg, train=False)

print(f'{DATASET} — pli {FOLD}')
print(f'  train : {len(train_df):>5} images / {train_df.patient_id.nunique()} patientes ({train_df.label.mean():.1%} malignes)')
print(f'  val   : {len(val_df):>5} images / {val_df.patient_id.nunique()} patientes ({val_df.label.mean():.1%} malignes)')

In [ ]:
# ── 6. Boucle de criblage ────────────────────────────────────────────────
# Chaque architecture est isolée dans un try/except : un échec (poids
# indisponibles, mémoire saturée) n'interrompt pas les 20 minutes de calcul
# des autres. L'erreur est consignée et le criblage continue.
import gc, traceback
import numpy as np
from src.models.encoders import MonoModalClassifier
from src.train.trainer import Trainer, TrainConfig
from src.eval.metrics import compute_metrics

results, predictions = [], {}
t_total = time.time()

for arch in ARCHITECTURES:
    bs = BATCH[arch]
    print(f"\n{'='*66}\n{arch}  (lot {bs})\n{'='*66}")

    for attempt in range(3):                     # 3 essais : bs, bs/2, bs/4
        try:
            torch.cuda.empty_cache(); gc.collect()
            t0 = time.time()

            train_loader = DataLoader(BreastDataset(train_df, ROOT, tf_train),
                                      batch_size=bs, shuffle=True,
                                      num_workers=2, pin_memory=True, drop_last=True)
            val_loader   = DataLoader(BreastDataset(val_df, ROOT, tf_val),
                                      batch_size=bs, shuffle=False, num_workers=2)

            model = MonoModalClassifier(encoder_name=arch, pretrained=True, image_size=224)
            n_params = sum(p.numel() for p in model.parameters()) / 1e6

            trainer = Trainer(model, train_loader, val_loader,
                              TrainConfig(epochs=EPOCHS, patience=PATIENCE, batch_size=bs,
                                          seed=SEED, run_name=f'{DATASET}_{arch}_f{FOLD}',
                                          out_dir=CKPT_DIR))
            summary = trainer.fit()

            trainer.load_best()
            scores, targets = trainer.predict(val_loader)
            m = compute_metrics(targets, scores)

            predictions[arch] = scores
            y_true = targets                      # identique pour toutes : DeLong applicable
            results.append({'architecture': arch, 'params_M': round(n_params, 2),
                            'batch': bs, 'auc': m['auc_roc'],
                            'sensibilite': m['sensitivity'], 'specificite': m['specificity'],
                            'f1': m['f1'], 'mcc': m['mcc'],
                            'spec_sens95': m['spec_at_sens95'],
                            'epoques': summary['epochs_run'],
                            'meilleure_epoque': summary['best_epoch'],
                            'minutes': round((time.time()-t0)/60, 2), 'statut': 'ok'})
            print(f"  → AUC {m['auc_roc']:.4f} | {n_params:.1f} M params | {(time.time()-t0)/60:.1f} min")
            break

        except torch.cuda.OutOfMemoryError:
            bs = max(4, bs // 2)
            print(f'  ⚠ mémoire saturée — nouvel essai avec un lot de {bs}')
            del model; torch.cuda.empty_cache(); gc.collect()
        except Exception as exc:
            print(f'  ❌ échec : {type(exc).__name__} — {exc}')
            traceback.print_exc(limit=2)
            results.append({'architecture': arch, 'params_M': None, 'auc': None,
                            'statut': f'échec: {type(exc).__name__}'})
            break
    else:
        results.append({'architecture': arch, 'auc': None, 'statut': 'mémoire insuffisante'})

    torch.cuda.empty_cache(); gc.collect()

print(f"\n{'='*66}\nCriblage terminé en {(time.time()-t_total)/60:.1f} min")

In [ ]:
# ── 7. Classement ────────────────────────────────────────────────────────
res = pd.DataFrame(results).sort_values('auc', ascending=False, na_position='last')
res = res.reset_index(drop=True)
display(res)

ok = res[res.statut == 'ok']
assert len(ok) >= 2, 'Trop peu d’architectures ont abouti pour comparer.'
print(f"\nMeilleure : {ok.iloc[0].architecture} (AUC {ok.iloc[0].auc:.4f}, {ok.iloc[0].params_M:.1f} M)")

In [ ]:
# ── 8. Test de DeLong : les écarts sont-ils RÉELS ? ──────────────────────
# Étape décisive. Sur ~110 images de validation, un écart d'AUC de 0,03 peut
# n'être que du bruit. Sans ce test, on choisirait une architecture au hasard
# en croyant l'avoir sélectionnée.
from src.eval.delong import pairwise_delong, auc_ci

print('Intervalles de confiance à 95 % :')
for a in ok.architecture:
    ci = auc_ci(y_true, predictions[a])
    print(f"  {a:<18} AUC {ci['auc']:.4f}  [{ci['ci_low']:.3f} ; {ci['ci_high']:.3f}]")

cmp = pairwise_delong(y_true, {a: predictions[a] for a in ok.architecture})
print('\nComparaisons deux à deux :')
display(cmp[['name_a', 'name_b', 'auc_a', 'auc_b', 'delta', 'p_value', 'significant']])

n_sig = int(cmp.significant.sum())
print(f"\n{n_sig} paire(s) significative(s) sur {len(cmp)}.")
if n_sig == 0:
    print('→ Aucune architecture ne se distingue statistiquement sur ce pli.')
    print('  Départagez sur le COÛT (paramètres, temps) — argument d’Alswilem & Pacal.')

In [ ]:
# ── 9. Compromis performance / complexité ────────────────────────────────
import matplotlib.pyplot as plt

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.5))

a1.scatter(ok.params_M, ok.auc, s=90, zorder=3)
for _, r in ok.iterrows():
    a1.annotate(r.architecture, (r.params_M, r.auc),
                textcoords='offset points', xytext=(6, 5), fontsize=9)
a1.set_xscale('log'); a1.set_xlabel('paramètres (M, échelle log)')
a1.set_ylabel('AUC'); a1.grid(alpha=0.3)
a1.set_title('Performance vs complexité')

a2.barh(ok.architecture, ok.minutes)
a2.set_xlabel('minutes'); a2.set_title('Coût d’entraînement'); a2.grid(alpha=0.3, axis='x')

plt.tight_layout(); plt.show()

corr = ok[['params_M', 'auc']].corr().iloc[0, 1]
print(f'Corrélation paramètres ↔ AUC : {corr:+.3f}')
print('→ Corrélation négative : les gros modèles sur-apprennent, comme attendu sur ce volume de données.'
      if corr < 0 else
      '→ Corrélation positive : la capacité aide encore. À revérifier à l’étage 2.')

In [ ]:
# ── 10. Décision et journalisation ───────────────────────────────────────
from src.train.trainer import log_run

TOP_N = 3
retenues = ok.head(TOP_N).architecture.tolist()

print('ÉTAGE 2 — architectures retenues pour la validation croisée 5 plis :')
for i, a in enumerate(retenues, 1):
    r = ok[ok.architecture == a].iloc[0]
    print(f"  {i}. {a:<18} AUC {r.auc:.4f}  ({r.params_M:.1f} M, {r.minutes:.1f} min)")
print('\nÉliminées :', ', '.join(ok.tail(len(ok)-TOP_N).architecture.tolist()) or '—')

res.to_csv(RESULTS_DIR / f'criblage_{DATASET}_fold{FOLD}.csv', index=False)
cmp.to_csv(RESULTS_DIR / f'criblage_{DATASET}_delong.csv', index=False)
np.savez(RESULTS_DIR / f'criblage_{DATASET}_predictions.npz',
         y_true=y_true, **{a: predictions[a] for a in ok.architecture})

for _, r in ok.iterrows():
    log_run(RESULTS_DIR, f"criblage_{DATASET}_{r.architecture}",
            config={'etape': 'criblage', 'dataset': DATASET, 'fold': FOLD,
                    'encoder': r.architecture, 'image_size': 224, 'seed': SEED},
            metrics={'auc_roc': r.auc, 'sensitivity': r.sensibilite,
                     'specificity': r.specificite, 'f1': r.f1, 'mcc': r.mcc},
            extra={'params_M': r.params_M, 'minutes': r.minutes})

print(f'\n✅ Résultats enregistrés dans {RESULTS_DIR}')

## Comment lire ces résultats

**Ne choisissez pas l'architecture sur la seule AUC.** Regardez la cellule 8 d'abord.

| Situation | Décision |
|---|---|
| Des paires sont significatives (p < 0,05) | Retenir les 3 meilleures pour l'étage 2 |
| **Aucune paire significative** | Les architectures sont équivalentes sur ce pli. Départager sur le **coût** : à performance égale, le modèle le plus léger gagne — c'est exactement l'argument d'Alswilem & Pacal [15] |
| Un modèle lourd domine nettement | Vérifier ses courbes de loss avant de conclure : un écart sur un seul pli reste fragile |

**Le graphique de la cellule 9** teste l'hypothèse de la section 6.3 du cahier des charges. Une corrélation négative entre paramètres et AUC confirmerait que la capacité excédentaire nuit sur de petits volumes — résultat en soi, à faire figurer dans le rapport.

**Attention à ne pas surinterpréter.** Ce criblage repose sur **un seul pli** d'environ 110 images de validation. C'est un instrument d'élimination, pas de sélection définitive : l'étage 2 (5 plis) tranchera.

## Étape suivante

1. Relancer ce notebook avec `DATASET = 'cbis_ddsm'` pour la mammographie — **beaucoup plus long** (2 857 images de très grande taille), à lancer sur une session dédiée.
2. Puis l'étage 2 : validation croisée 5 plis sur les 3 architectures retenues, par modalité.
3. L'architecture retenue par modalité sera ensuite **figée** pour toutes les expériences de fusion (Phase 3).